# Prototipo RAG: Consulta y Visualización de datos SIES

Nombre: Matías Zamorano

Comision:

Fecha: Marzo 2026

## 1. Introducción

**Nombre del proyecto**
Asistente Inteligente para la extracción de datos SIES

**Presentación del problema a abordar**
Los datos del Servicio de Información de Educación Superior (SIES) son fundamentales para entender el contexto academico en Chile (oferta, matrícula, titulación). Sin embargo, presentan un barrera de entrada tecnica:

- Requiere conocimientos en manejo de bases de datos estrucuturados

- Aplicación de procesos de filtrado y manipulación para su compresión.

**Desarrollo de la propuesta de solución**

Se propone implementar un sistema basado en IA generativa con arquitectura RAG. El sistema funciona bajo la premisa de un contexto controlado:

1.- El usuario hace una consulta en lenguaje natural.

2.- El sistema recupera los datos del SIES.

3.- Un LLM interpreta los datos y genera una respuesta estructura o una visualización.

**Justificación de la viabilidad del proyecto**

Desde el apartado tecnico el proyecto no requiere entrenamiento de modelos, sino el uso de APIs existentes y tecnicas de prompting.

En cuanto el consumo de recursos, al trabajar con datos publicos y con subconjunto acotado se asegura rapidez y calidad en las respuestas.

Considerar que para este prototipo los datos de uso de cargaran de forma local, a modo de simulación.

## 2. Objetivos del producto

**General**

Desarrollar una interfaz de lenguaje natural que permita consultar indicadores de educación superior sin conocimientos tecnicos.

**Especificos**

* Implementar técnicas de *Few-Shot prompting* para estandarizar las respuestas de datos.

* Optimizar la generación de visualizaciones mediante prompts estructurados

* Reducirlas alucinaciones del modelo obligándolo a trabajar exclusivamente con el contexto recuperado.

## 3. Metodología

Para lograr una solución robusta, se aplicará el siguiente procedimiento:

1.- **Manipulación de los datos:** seleecionar el grupo de archivos *CSV* del SIES, acoplarlos en una base de datos reducida y la conversión de la información buscando en  formatos legibles por el modelo sea *Markdown* o *JSON*.

2.- **Ingeniería de prompts:** diseño de una jerarquía de prompts para asegurar la calidad de los resultados. Considerando los siguientes pasos: interpretación, extracción y visualización.

3.- **Validación iterativa:** se realizan pruebas de los prompts con versión *Zero-shot* y *Few-shot* para comprar la precisión de los resultados generados.

4.- **Visualización:** transformación de datos tabulares en instrucciones para el diseño que se entregan a los modelos de generación de imagenes.

## 4. Herramientas y tecnologías

* **Modelos**: Gemini 1.5 flash para la consultas a los datos, dada la familiariadad con el modelo y si amplia ventana de contexto que facilita el trabajo con tablas de datos. Por su parte, DALL-3 se utilizara para generar las visualizaciones. Mencionar que la eleccion de Gemini es por la familiaridad con el uso de este modelo, mientras que DALL-3 se presenta como *placeholder* hasta definir si existe una mejor alternativa.

* **Tecnicas de Prompting**

  1.- Role promting: para definir al modelo como un "Analista de datos experto en educación superior chilena" para ajustar el tono y precisión de las respuestas.

  2.- Few-shot prompting: para asegurar que las respuetas tiene el formato deseado por los usuarios.

  3.- Demilitadores: uso de comillas o etiquetas para separar las variables de la base de datos que se quieren analizar según las necesidades del usuario.

## 5. Implementación

### Setup

Instalacion locala de las dependencias de IA

In [ ]:
!pip install -q -U google-generativeai openai pandas pillow requests

Carga de los modulos y APIKEYs

In [ ]:
import google.generativeai as genai
from openai import OpenAI
import pandas as pd
import os
from IPython.display import Image, display

# APIKEYs
GOOGLE_API_KEY = "GOOGLE_API_KEY"
OPENAI_API_KEY = "OPENAI_API_KEY"
#Dado que no es requisito de la entrega realizar la conexión a la API, solamente se deja explicito el codigo para realizarla.

genai.configure(api_key=GOOGLE_API_KEY)
client_oa = OpenAI(api_key=OPENAI_API_KEY)

### Carga de los datos

In [ ]:
df = pd.read_csv('datos_sies.csv', sep=';', encoding='latin-1')
#De momento no se carga la base de datos que se va a utilzar, dado que las consultas no hace por desde este notebook.

print("Datos cargados exitosamente. Columnas:", df.columns.tolist())

Muestra una muestra para verificar

In [ ]:
df.head()

### Solicitud texto-texto

Definimos el rol que tendra el modelo.

In [ ]:
system_prompt = """
Eres un Analista de Datos especializado en educación superior chilena.
IMPORTANTE: Solo puedes responder utilizando los datos proporcionados en el contexto.
Si la información no está en los datos, responde: 'Información no disponible en la base de datos actual'.
Formato de salida: Siempre incluye una tabla Markdown y un análisis de 2 líneas.
Mantén un tono profesional, objetivo y analítico.
"""

Ahora se definen los ejemplos de respuestas esperados por el usuario.

In [ ]:
few_shots = """
### Ejemplos del formato de respuestas ###
Ejemplo 1:
Datos: Año: 2024, Universidad: U. de Chile, Carrera: Ingeniería Civil, Vacantes: 100, Matriculados: 105
Respuesta: La carrera de Ingeniería Civil en la U. de Chile presenta una sobreocupación del 105% (105 matriculados para 100 vacantes).

Ejemplo 2:
Datos: Año: 2024, Universidad: IP Los Leones, Carrera: Diseño, Vacantes: 50, Matriculados: 20
Respuesta: Diseño en IP Los Leones muestra una baja demanda, cubriendo solo el 40% de sus vacantes disponibles.
"""

Se define la función para llamar al modelo

In [ ]:
def consultar_asistente_sies(datos_recuperados, pregunta_usuario):
  """
  Ensambla el prompt maestro combinando Sistema + Ejemplos + Datos + Pregunta.
  """

  # Ensamblaje del prompt final
  prompt_maestro = f"""
    {system_prompt}

    {few_shots}

    ---
    ### DATOS REALES ###
    {datos_recuperados}

    Pregunta: {pregunta_usuario}
  """
  model = genai.GenerativeModel('gemini-1.5-flash')

  response = model.generate_content(prompt_maestro)
  return response.text

Ejecución

In [ ]:
# Simulamos que el usuario pregunta por una carrera específica
carrera_buscada = "Ingeniería Informática"

# Filtramos el DataFrame localmente
filtro = df[df['Carrera'].str.contains(carrera_buscada, case=False)].head(3)
contexto_para_ia = filtro.to_markdown()
# Convertimos a formato tabla para el prompt

# Ejecución de la función
prompt_final_generado = consultar_asistente_sies(contexto_para_ia, f"Analiza la situación de {carrera_buscada}")

print("Prompt final: \n")
print(prompt_final_generado)

### Solicitud texto-imagen

Definimos el rol que tendra el modelo.

In [ ]:
system_prompt_vis = """
  Actúa como un experto en Visualización de Datos.
  Tu objetivo es transformar datos del SIES en instrucciones precisas para generar un gráfico.
  REGLAS:
    - Mantén un estilo profesional, limpio y minimalista (estilo institucional).
    - Usa terminología técnica de ejes (X, Y), etiquetas y títulos.
    - No incluyas datos que no estén en el contexto proporcionado.
  """

Se definen los formatos de respuesta esperados por el usuario.

In [ ]:
few_shot_vis = """
  ### Ejemplos de visualizaciones###

  Ejemplo 1:
  Input Datos: Universidad A: 500 matriculados, Universidad B: 300 matriculados.
  Estilo: Gráfico de barras moderno.
  Output Instrucciones:
    - Tipo: Bar Chart 2D.
    - Eje Vertical: "Total de Matriculados" (0 a 600).
    - Eje Horizontal: "Instituciones" (Universidad A, Universidad B).
    - Colores: Azul marino (#003366) y Gris claro.
    - Título: Comparativa de Matrícula Inicial 2024.

  Ejemplo 2:
  Input Datos: Carrera X: 80 vacantes vs 100 matriculados.
  Estilo: Infografía minimalista.
  Output Instrucciones:
    - Concepto: Comparación de brecha (Vacantes vs Real).
    - Elementos: Dos barras paralelas por categoría.
    - Etiquetas: "Capacidad" (80) y "Demanda" (100).
    - Título: Análisis de Sobre-cupo Carrera X.
"""

Se define la función para llamar al modelo

In [ ]:
def generar_visualizacion_sies(datos_resumidos, estilo_visual):
    """
    Función para transformar datos tabulares en instrucciones
    detalladas de visualización (Texto-a-Imagen/Gráfico).
    """
    # Construccion del prompt maestro
    prompt_vis_maestro = f"""
    {system_prompt_vis}

    {few_shot_vis}

    ### DATOS REALES PARA VISUALIZAR ###
    {datos_resumidos}

    Estilo solicitado por el usuario: {estilo_visual}

    Genera las instrucciones detalladas para el modelo de imagen:
  """
  model = genai.GenerativeModel('gemini-1.5-flash')

  response = model.generate_content(prompt_maestro)
  return response.text

Ejecución

In [ ]:
datos_input = "Ingeniería Informática: 120 matriculados, Ingeniería Comercial: 200 matriculados"
estilo_input = "Gráfico de barras corporativo"

prompt_resultado = generar_visualizacion_sies(datos_input, estilo_input)
print(prompt_resultado)